# 04 — Node Behavior Feature Engineering


## Objective
Assemble the complete numeric feature matrix used by all subsequent anomaly detectors.

**Feature families**
1. Transaction-level (purchase_value, age, …)
2. Temporal (account_age, hour, instant_purchase, …)
3. Behavioural / frequency (user/device/IP transaction counts)
4. Graph / structural (degree, users-on-device, users-on-ip, …)
5. Low-cardinality categorical one-hots (source, browser, sex, country)

The fraud label is explicitly dropped and never used.


In [ ]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data_utils import load_processed, save_processed
from graph_features import build_feature_matrix, FEATURE_CANDIDATES

train = load_processed("train")
val   = load_processed("val")
test  = load_processed("test")

print("Building feature matrices (train / val / test) …")
X_train, feats = build_feature_matrix(train)
X_val, _       = build_feature_matrix(val)
X_test, _      = build_feature_matrix(test)

# Align columns
all_cols = sorted(set(X_train.columns) | set(X_val.columns) | set(X_test.columns))
for X in (X_train, X_val, X_test):
    for c in all_cols:
        if c not in X.columns:
            X[c] = 0.0
X_train = X_train[all_cols]
X_val   = X_val[all_cols]
X_test  = X_test[all_cols]

print("Feature matrix shape (train):", X_train.shape)
print("Number of features:", len(all_cols))
print("\nFeature list (first 30):", all_cols[:30])

# Persist
save_processed(X_train.assign(class=train["class"].values), "X_train")
save_processed(X_val.assign(class=val["class"].values), "X_val")
save_processed(X_test.assign(class=test["class"].values), "X_test")
print("Saved feature matrices with label attached for later validation only.")
